[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch10-model-compression/04-byte-bang-thong-va-ti-so.ipynb)

# Byte, băng thông và tỉ số trên giấy: khi nào mô hình nén thật sự nhanh hơn

Ba sổ tay trước làm mô hình nhỏ đi: ít trọng số khác 0 hơn, ít tham số hơn, ít bit hơn. Sổ tay
này hỏi câu chương 10 hỏi sau mỗi kỹ thuật: những tỉ số trên giấy ấy đổi thành gì trên một thiết
bị thật? Phần lớn là phép tính trên số liệu của chương, vì điện thoại, vi điều khiển và GPU A100
mà chương nói tới không có ở đây. Phần 3 đo trên chính CPU bạn đang chạy.

**Bạn sẽ làm:**
1. đặt các mô hình đại diện của chương cạnh dung lượng thiết bị (bảng 2, napkin math 1.1), và tìm
   xem bảng chia theo đơn vị nào;
2. tính napkin math 1.4: byte của trọng số và của bộ nhớ đệm KV (KV cache), rồi số token mỗi giây
   khi suy luận (inference) nghẽn ở bộ nhớ (memory-bound);
3. đo một tầng 4096 nhân 4096 trên CPU này ở fp32, bf16 và int8, rồi theo kích thước batch
   (batch size);
4. tính phía tính toán: napkin math 1.2, 1.3 và thang năng lượng của chương;
5. đếm lưu lượng bộ nhớ của Conv-BN-ReLU trước và sau hợp nhất toán tử (operator fusion);
6. áp định luật Amdahl (Amdahl's law), rồi đặt tỉ số trên giấy cạnh kết quả trong các kịch bản
   của chương.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Khoảng cách triển khai: byte của mô hình và dung lượng của thiết bị

Chương mở đầu bằng một phép tính không cần kỹ thuật nào: một mô hình ngôn ngữ 7 tỷ tham số ở FP16
cần 14 GB chỉ để chứa trọng số, còn điện thoại có 8 GB RAM dùng chung với hệ điều hành và các ứng
dụng. Kích thước mô hình là số tham số nhân số byte mỗi giá trị, nên mọi phép tính dưới đây đều
là phép nhân như thế. Có hai cách đọc đơn vị:

* thập phân: 1 GB là $10^9$ byte, 1 KB là $10^3$ byte;
* nhị phân: 1 GiB là $2^{30}$ byte, 1 KiB là $2^{10}$ byte.

In [ ]:
GB, MB, KB = 1e9, 1e6, 1e3        # thập phân
GiB, KiB = 2 ** 30, 2 ** 10       # nhị phân

BIT_FP32 = sach(32, nguon="32 · 16 · 8 bit")
BIT_FP16 = sach(16, nguon="32 · 16 · 8 bit")
BIT_INT8 = sach(8, nguon="32 · 16 · 8 bit")
theo_sach("FP32 sang INT8: số byte thô mỗi giá trị giảm", BIT_FP32 / BIT_INT8, sach=4, nguon="4 lần")

n_7b = sach(7e9, nguon="7 tỷ · 14 GB · 8 GB")
ram_dien_thoai = sach(8, nguon="7 tỷ · 14 GB · 8 GB")
w_7b = n_7b * BIT_FP16 / 8 / GB
theo_sach("7 tỷ tham số ở FP16", w_7b, sach=14, nguon="7 tỷ · 14 GB · 8 GB")
print(f"   vừa {ram_dien_thoai} GB RAM của điện thoại? {w_7b <= ram_dien_thoai}")

n_175b = sach(175e9, nguon="175 tỷ · hơn 350 GB · 512 KB")
sram_vdk = sach(512, nguon="175 tỷ · hơn 350 GB · 512 KB")
theo_sach("175 tỷ tham số ở FP16", n_175b * BIT_FP16 / 8 / GB, sach=350, nguon="350 GB → 175 GB · 2×")
bac = math.log10(n_175b * BIT_FP16 / 8 / (sram_vdk * KB))
print(f"   so với {sram_vdk} KB SRAM của một vi điều khiển: cách nhau {bac:.1f} bậc độ lớn")
theo_sach("175 tỷ tham số ở INT8", n_175b * BIT_INT8 / 8 / GB, sach=175, nguon="350 GB → 175 GB · 2×")
theo_sach("FP16 sang INT8", BIT_FP16 / BIT_INT8, sach=2, nguon="350 GB → 175 GB · 2×");

Gần sáu bậc độ lớn, đúng như chương nói. Không kỹ thuật đơn lẻ nào lấp được khoảng ấy, nên chương
coi nén mô hình là một ngành kỹ thuật có hệ thống, không phải một mẹo.

Bảng 2 đặt sáu mô hình đại diện cạnh ba loại thiết bị: đám mây (khoảng 107 GB), điện thoại (8 GB)
và TinyML (khoảng 512 KB), rồi ghi mỗi mô hình vượt dung lượng bao nhiêu lần.

### Dự đoán

DLRM có 100 GB trọng số, điện thoại có 8 GB. Theo bảng 2, DLRM vượt điện thoại 11.6 lần. Bạn có ra
đúng số ấy bằng một phép chia không? Và "khoảng 107 GB" của đám mây là con số tròn ở đơn vị nào?

In [ ]:
# Bảng 2: trọng số của sáu mô hình, byte thập phân như sách ghi
mo_hinh = {
    "DLRM": sach(100, nguon="100 GB · 6 GB · 102.4 MB") * GB,
    "GPT-2 XL": sach(6, nguon="100 GB · 6 GB · 102.4 MB") * GB,
    "ResNet-50": sach(102.4, nguon="100 GB · 6 GB · 102.4 MB") * MB,
    "MobileNetV2": sach(14, nguon="102.4 MB · 14 MB · 3.5 MB") * MB,
    "MobileNetV2 INT8": sach(3.5, nguon="14 MB · 3.5 MB · 200 KB") * MB,
    "DS-CNN INT8": sach(200, nguon="3.5 MB · 200 KB") * KB,
}
# Ba thiết bị, đọc theo nhị phân. Đám mây 100 GiB là giả thuyết của sổ tay, kiểm ngay bên dưới.
thiet_bi = {
    "đám mây": 100 * GiB,
    "điện thoại": ram_dien_thoai * GiB,
    "TinyML": sram_vdk * KiB,
}
theo_sach("dung lượng đám mây, 100 GiB đổi ra GB", thiet_bi["đám mây"] / GB, sach=107,
          nguon="~107 GB · 8 GB", tol=0.5)


def vuot(m, tb):
    return mo_hinh[m] / thiet_bi[tb]


theo_sach("DLRM so với điện thoại", vuot("DLRM", "điện thoại"), sach=11.6, nguon="11.6× · 190,734.9×")
theo_sach("DLRM so với TinyML", vuot("DLRM", "TinyML"), sach=190_734.9, nguon="11.6× · 190,734.9×")
theo_sach("GPT-2 XL so với TinyML", vuot("GPT-2 XL", "TinyML"), sach=11_444.1, nguon="190,734.9× · 11,444.1×")
theo_sach("ResNet-50 so với TinyML", vuot("ResNet-50", "TinyML"), sach=195.3, nguon="11,444.1× · 195.3×")
theo_sach("MobileNetV2 so với TinyML", vuot("MobileNetV2", "TinyML"), sach=26.7, nguon="195.3× · 26.7×")
theo_sach("MobileNetV2 INT8 so với TinyML", vuot("MobileNetV2 INT8", "TinyML"), sach=6.7, nguon="26.7× · 6.7×")

print(f"\n{'':<18}" + "".join(f"{tb:>17}" for tb in thiet_bi))
for m in mo_hinh:
    o = ["vừa" if vuot(m, tb) <= 1 else f"vượt {vuot(m, tb):,.1f}×" for tb in thiet_bi]
    print(f"{m:<18}" + "".join(f"{x:>17}" for x in o))

print("\nNếu đọc dung lượng thiết bị theo thập phân:")
print(f"   DLRM so với điện thoại: {mo_hinh['DLRM'] / (ram_dien_thoai * GB):.1f}×")
print(f"   DLRM so với TinyML: {mo_hinh['DLRM'] / (sram_vdk * KB):,.1f}×")


# Hình: trọng số của từng mô hình và dung lượng của từng thiết bị, thang log
def nhan_byte(b):
    for don_vi, ten in [(GB, "GB"), (MB, "MB"), (KB, "KB")]:
        if b >= don_vi:
            return f"{b / don_vi:g} {ten}"


fig, ax = plt.subplots(figsize=(7.5, 3.8))
ds = list(mo_hinh)[::-1]
ax.barh(range(len(ds)), [mo_hinh[m] for m in ds], color="0.78", height=0.6, left=1e4)
ax.set_yticks(range(len(ds)), ds, fontsize=9)
ax.set_xscale("log")
ax.set_xlim(1e4, 1e12)
mau_tb = {"đám mây": "C0", "điện thoại": "C1", "TinyML": "C3"}
for tb, c in thiet_bi.items():
    ax.axvline(c, color=mau_tb[tb], ls="--", lw=1.4)
    ax.text(c, 1.01, tb, color=mau_tb[tb], transform=ax.get_xaxis_transform(),
            ha="center", va="bottom", fontsize=9)
for i, m in enumerate(ds):
    ax.text(mo_hinh[m] / 1.25, i, nhan_byte(mo_hinh[m]), ha="right", va="center", fontsize=8)
i_mb = ds.index("MobileNetV2 INT8")
ax.annotate("", xy=(thiet_bi["TinyML"], i_mb + 0.42), xytext=(mo_hinh["MobileNetV2 INT8"], i_mb + 0.42),
            arrowprops=dict(arrowstyle="<->", color="C3", lw=1))
ax.text(1.6e6, i_mb + 0.48, "sách: 6.7×", color="C3", fontsize=8, va="bottom")
i_dl = ds.index("DLRM")
ax.annotate("", xy=(thiet_bi["điện thoại"], i_dl - 0.42), xytext=(mo_hinh["DLRM"], i_dl - 0.42),
            arrowprops=dict(arrowstyle="<->", color="C1", lw=1))
ax.text(2.6e10, i_dl - 0.48, "sách: 11.6×", color="C1", fontsize=8, va="top", ha="center")
ax.set_xlabel("byte (thang log)")
ax.set_title("Bảng 2: trọng số của mô hình và dung lượng của thiết bị", fontsize=10, pad=18)
ax.grid(True, axis="y", alpha=0)
plt.tight_layout()
plt.show()

Một phép chia thẳng 100 cho 8 cho 12.5, không phải 11.6. Các tỉ số của bảng 2 chỉ khớp khi kích
thước mô hình đọc theo thập phân còn dung lượng thiết bị đọc theo nhị phân: điện thoại 8 GiB,
vi điều khiển 512 KiB, và "khoảng 107 GB" của đám mây chính là 100 GiB. Hai cách đọc lệch nhau
chừng 7 phần trăm ở cỡ GB và 2.4 phần trăm ở cỡ KB. Không ô "vừa" hay "vượt" nào của bảng đổi
vì thế, nhưng khi bạn tự tính lại một tỉ số của sách mà lệch ở chữ số thứ hai, hãy thử cách đọc
đơn vị kia trước khi nghi phép tính.

### Hai thiết bị chật: chuông cửa và một GPU tiêu dùng

Hai ví dụ khác của chương cho thấy lượng tử hoá (quantization) quyết định mô hình có chạy được
hay không, trước cả câu hỏi về tốc độ. Mô hình phát hiện từ khoá (keyword spotting) DS-CNN có
800 KB trọng số ở FP32, vượt ngân sách SRAM khoảng 512 KB. Một vi điều khiển chuông cửa thông minh
có 256 KB SRAM, và sau bộ đệm của môi trường chạy (runtime), cửa sổ âm thanh và phần trích đặc
trưng thì chỉ còn chừng 100 KB cho trọng số. Napkin math 1.1 làm cùng phép tính cho Llama 3 8B.

In [ ]:
ds_cnn_fp32 = sach(800, nguon="800 KB · 200 KB · 256 KB")
ngan_sach_tinyml = sach(512, nguon="~512 KB · 800 KB", tol=0.5)
ngan_sach_chuong_cua = sach(100, nguon="256 KB · 100 KB · 4×")
BIT_NHI_PHAN = sach(1, nguon="1 bit · 2 bit")
theo_sach("DS-CNN ở INT8", ds_cnn_fp32 * BIT_INT8 / BIT_FP32, sach=200, nguon="800 KB · 200 KB")
theo_sach("FP32 so với INT8", BIT_FP32 / BIT_INT8, sach=4, nguon="256 KB · 100 KB · 4×")
for ten, bit in [("FP32", BIT_FP32), ("INT8", BIT_INT8), ("INT4", 4), ("nhị phân", BIT_NHI_PHAN)]:
    kb = ds_cnn_fp32 * bit / BIT_FP32
    print(f"   DS-CNN {ten:>9}: {kb:6.1f} KB · TinyML {'vừa' if kb <= ngan_sach_tinyml else 'không vừa':<9}"
          f" · chuông cửa {'vừa' if kb <= ngan_sach_chuong_cua else 'không vừa'}")

print()
n_llama = sach(8e9, nguon="8 tỷ · 2 byte · 16 GB")
w_llama16 = n_llama * sach(2, nguon="8 tỷ · 2 byte") / GB
theo_sach("Llama 3 8B ở FP16", w_llama16, sach=16, nguon="2 byte · 16 GB · 24 GB")
print(f"   16 GB trọng số không để lại dư địa: thiết bị lớp {sach(24, nguon='16 GB · 24 GB')} GB mới thực tế")
w_llama4 = n_llama * sach(0.5, nguon="0.5 byte · 4 GB") / GB
theo_sach("Llama 3 8B ở INT4", w_llama4, sach=4, nguon="0.5 byte · 4 GB")
theo_sach("phần còn lại trên thiết bị 8 GB", sach(8, nguon="4 GB · 8 GB") - w_llama4, sach=4,
          nguon="0.5 byte · 4 GB · 8 GB");

Ở chuông cửa, INT8 không còn đủ: 200 KB gấp đôi phần dành cho trọng số. INT4 vừa đúng sát mép, và
đó là lý do chương nói thiết bị này đẩy người ta tới INT4 hay nhị phân. Còn ở Llama 3 8B, 4 GB còn
lại phải chứa siêu dữ liệu, bộ nhớ đệm KV, giá trị kích hoạt (activation) và vùng làm việc. Phần 2
cho thấy bộ nhớ đệm KV không nhỏ chút nào.

## 2. Napkin math 1.4: khi mỗi token phải đọc lại cả mô hình

Khi sinh văn bản theo kiểu tự hồi quy (autoregressive) với batch 1, mỗi token mới phải nạp toàn bộ
trọng số từ bộ nhớ, trong khi mỗi byte nạp về chỉ dùng cho rất ít phép tính. Thời gian một token
khi đó gần bằng số byte phải đọc chia cho băng thông (bandwidth). Chương dùng một **mô hình lưu
lượng đơn giản hoá**: mỗi token đọc toàn bộ trọng số cộng toàn bộ bộ nhớ đệm KV.

Kịch bản: mô hình 7 tỷ tham số ở FP16, thiết bị 16 GB RAM, ngữ cảnh 4,096 token, 32 tầng, 32 head
mỗi tầng, 128 chiều mỗi head, băng thông 50 GB/s. Bộ nhớ đệm KV giữ khoá và giá trị của mọi token
trước đó, ở mọi tầng:

$$\text{KV} = 2 \times \text{số tầng} \times \text{số head} \times \text{số chiều mỗi head}
\times \text{số byte mỗi giá trị} \times \text{số token}$$

### Dự đoán

Lượng tử hoá trọng số từ FP16 xuống INT4 làm trọng số nhỏ đi 4 lần. Tốc độ sinh token tăng đúng 4
lần, ít hơn hay nhiều hơn? Vì sao?

In [ ]:
n = sach(7e9, nguon="7 tỷ · 16 GB · 14 GB")
RAM = sach(16, nguon="7 tỷ · 16 GB · 14 GB")
SO_TOKEN = sach(4096, nguon="4,096 token · 32 tầng")
SO_TANG = sach(32, nguon="4,096 token · 32 tầng")
SO_HEAD = sach(32, nguon="32 head · 128 chiều")
CHIEU_HEAD = sach(128, nguon="32 head · 128 chiều")
BW = sach(50, nguon="50 GB/s · 323 ms")
BYTE_FP16 = BIT_FP16 / 8
BYTE_INT4 = sach(0.5, nguon="0.5 byte · 4 GB")


def bo_nho_kv(so_token, byte_moi_gt=BYTE_FP16):
    return 2 * SO_TANG * SO_HEAD * CHIEU_HEAD * byte_moi_gt * so_token   # 2: khoá và giá trị


def mot_token(byte_trong_so, so_token):
    """Mô hình lưu lượng của chương: (số byte phải đọc, số giây) cho một token."""
    tong = n * byte_trong_so + bo_nho_kv(so_token)
    return tong, tong / (BW * GB)


theo_sach("trọng số FP16", n * BYTE_FP16 / GB, sach=14, nguon="16 GB · 14 GB")
theo_sach("bộ nhớ đệm KV ở 4,096 token", bo_nho_kv(SO_TOKEN) / GB, sach=2.1, nguon="128 chiều · 2.1 GB")
tong16, t16 = mot_token(BYTE_FP16, SO_TOKEN)
theo_sach("tổng bộ nhớ, FP16", tong16 / GB, sach=16.1, nguon="2.1 GB · 16.1 GB")
print(f"   vừa {RAM} GB RAM, chưa kể hệ điều hành? {tong16 / GB <= RAM}")
theo_sach("thời gian mỗi token, FP16", 1e3 * t16, sach=323, nguon="50 GB/s · 323 ms")
theo_sach("token mỗi giây, FP16", 1 / t16, sach=3.1, nguon="323 ms · 3.1 token mỗi giây")
print()
tong4, t4 = mot_token(BYTE_INT4, SO_TOKEN)
theo_sach("trọng số INT4", n * BYTE_INT4 / GB, sach=3.5, nguon="3.5 GB · 5.6 GB")
theo_sach("tổng bộ nhớ, trọng số INT4", tong4 / GB, sach=5.6, nguon="3.5 GB · 5.6 GB")
theo_sach("thời gian mỗi token, INT4", 1e3 * t4, sach=113, nguon="5.6 GB · 113 ms")
theo_sach("token mỗi giây, INT4", 1 / t4, sach=9, nguon="113 ms · 9 token mỗi giây")
theo_sach("tăng tốc", t16 / t4, sach=2.9, nguon="9 token mỗi giây · 2.9×")
print()
print(f"bộ nhớ đệm KV chiếm {bo_nho_kv(SO_TOKEN) / tong16:.0%} lưu lượng mỗi token ở FP16, "
      f"{bo_nho_kv(SO_TOKEN) / tong4:.0%} khi trọng số là INT4")
t4_ca_kv = (n * BYTE_INT4 + bo_nho_kv(SO_TOKEN, BYTE_INT4)) / (BW * GB)
print(f"nếu bộ nhớ đệm KV cũng xuống INT4: tăng tốc {t16 / t4_ca_kv:.2f}×")

Tăng tốc ít hơn 4 lần, vì bộ nhớ đệm KV vẫn ở FP16 và không nhỏ đi. Trọng số càng nhỏ thì phần
không đổi ấy càng chiếm nhiều lưu lượng. Sách viết "khoảng 2.1 GB" cho bộ nhớ đệm KV và 9 token
mỗi giây: cả hai là số đã làm tròn, nên dòng tương ứng ở trên lệch ở chữ số cuối mà vẫn khớp.

Ngữ cảnh dài hơn thì bộ nhớ đệm KV lớn lên tuyến tính, còn trọng số thì không. Hình dưới quét độ
dài ngữ cảnh với ba độ chính xác của trọng số, bộ nhớ đệm KV luôn ở FP16.

In [ ]:
ngu_canh = 2 ** np.arange(8, 16)          # 256 tới 32,768 token
fig, ax = plt.subplots(figsize=(7, 4.4))
for ten, b, mau in [("trọng số FP16", BYTE_FP16, "C0"), ("trọng số INT8", BIT_INT8 / 8, "C1"),
                    ("trọng số INT4", BYTE_INT4, "C2")]:
    tong = np.array([mot_token(b, int(L))[0] for L in ngu_canh])
    tps = 1 / (tong / (BW * GB))
    ax.plot(ngu_canh, tps, color=mau, lw=1.8, marker="o", ms=4, label=ten)
    qua = tong / GB > RAM
    ax.plot(ngu_canh[qua], tps[qua], "o", ms=8, mfc="white", color=mau)
ax.plot([], [], "o", ms=8, mfc="white", color="0.4", label=f"vượt {RAM} GB RAM")
ax.annotate("sách: 3.1 token/s", xy=(SO_TOKEN, 1 / t16), xytext=(SO_TOKEN / 9, 1.0),
            fontsize=8, arrowprops=dict(arrowstyle="->", color="0.3"))
ax.annotate("sách: 9 token/s", xy=(SO_TOKEN, 1 / t4), xytext=(SO_TOKEN * 2.2, 11),
            fontsize=8, arrowprops=dict(arrowstyle="->", color="0.3"))
ax.set_xscale("log", base=2)
ax.set_xticks(ngu_canh, [f"{L:,}" for L in ngu_canh], fontsize=8)
ax.set_xlabel("độ dài ngữ cảnh (token, thang log)")
ax.set_ylabel("token mỗi giây ở 50 GB/s")
ax.set_ylim(0, None)
ax.set_title("Mô hình 7B: bộ nhớ đệm KV ăn dần phần lợi của trọng số nhỏ", fontsize=10)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=4, fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## 3. Đo trên CPU này: một tầng 4096 nhân 4096, từng vector một

Phần 2 giả định thời gian tỉ lệ với số byte phải đọc. Ta kiểm giả định ấy trên một tầng kết nối
đầy đủ (fully connected layer) 4096 nhân 4096, đúng cỡ một phép chiếu của mô hình 7B ở phần 2
(32 head nhân 128 chiều): nhân ma trận trọng số với **một** vector, như khi sinh một token. Phép
nhân đi qua `F.linear`, đúng đường `nn.Linear` dùng. Ba cách lưu trọng số:

1. fp32, 4 byte mỗi trọng số;
2. bf16, 2 byte; CPU tính thẳng trên kiểu này nếu nó có lệnh và kernel cho bf16;
3. int8 với một hệ số co giãn cho mỗi hàng, 1 byte. PyTorch trên CPU không có kernel nhân ma trận
   int8 với một vector số thực, nên ta giải lượng tử hoá từng khối 256 hàng sang fp32 rồi nhân. Hệ
   số co giãn của hàng $i$ nhân vào **sau** tổng, vì $y_i = s_i \sum_j q_{ij} x_j$.

Dòng thứ tư là cùng phép nhân fp32 ấy, gọi qua `torch.mv`: cùng số byte, cùng phép tính, chỉ khác
kernel mà thư viện chọn.

### Dự đoán

Nếu phép nhân này nghẽn ở bộ nhớ, thời gian của bf16 và của int8 bằng bao nhiêu phần thời gian của
fp32? Còn nếu chính việc đổi kiểu dữ liệu tốn đáng kể thì sao? Và hai dòng fp32 có mất cùng thời
gian không?

In [ ]:
F = torch.nn.functional
D = 4096       # 32 head nhân 128 chiều
KHOI = 256     # số hàng giải lượng tử hoá mỗi lần

dat_hat_giong()
W = torch.randn(D, D) * 0.02
x = torch.randn(1, D)          # một token: batch 1
W_bf16, x_bf16 = W.to(torch.bfloat16), x.to(torch.bfloat16)
s_hang = W.abs().amax(dim=1) / 127
W_int8 = torch.round(W / s_hang[:, None]).clamp(-127, 127).to(torch.int8)


def nhan_int8(Wq, s, v):
    y = torch.empty(v.shape[0], Wq.shape[0])
    for i in range(0, Wq.shape[0], KHOI):
        y[:, i:i + KHOI] = F.linear(v, Wq[i:i + KHOI].float()) * s[i:i + KHOI]
    return y


def do_thoi_gian(f, lap=21):
    f()
    ts = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts))


y_ref = F.linear(x, W)
sai = (nhan_int8(W_int8, s_hang, x) - y_ref).abs().max() / y_ref.abs().max()
print(f"sai số lớn nhất của int8 so với fp32: {sai:.2%} của giá trị lớn nhất")
assert sai < 0.05

cach = {   # tên: (hàm, số byte trọng số phải đọc)
    "fp32": (lambda: F.linear(x, W), W.numel() * 4),
    "bf16": (lambda: F.linear(x_bf16, W_bf16), W.numel() * 2),
    "int8, giải lượng tử hoá\ntừng khối rồi nhân fp32": (lambda: nhan_int8(W_int8, s_hang, x),
                                                       W.numel() * 1 + 4 * D),
    "fp32, cùng phép nhân\nqua torch.mv": (lambda: torch.mv(W, x[0]), W.numel() * 4),
}
t_do = {ten: do_thoi_gian(f) for ten, (f, _) in cach.items()}
t_doi_kieu = do_thoi_gian(lambda: W_int8.float())
for ten, (_, so_byte) in cach.items():
    do_tren_may(f"{ten.replace(chr(10), ' ')} ({so_byte / MB:.1f} MB trọng số): thời gian", 1e3 * t_do[ten], "ms")
do_tren_may("chỉ riêng đổi cả ma trận int8 sang fp32: thời gian", 1e3 * t_doi_kieu, "ms")
for ten, (_, so_byte) in cach.items():
    do_tren_may(f"{ten.replace(chr(10), ' ')}: số byte trọng số chia thời gian", so_byte / t_do[ten] / GB, "GB/s")

# Hình: thời gian đo được, và thời gian "nếu tỉ lệ với byte" tính từ dòng fp32 đầu tiên
ten_cach = list(cach)
fig, ax = plt.subplots(figsize=(7, 3.6))
yy = np.arange(len(ten_cach))[::-1]
ax.barh(yy, [1e3 * t_do[k] for k in ten_cach], color=["C0", "C1", "C2", "C0"], height=0.55)
ti_le = [cach[k][1] / cach["fp32"][1] for k in ten_cach]
ax.plot([1e3 * t_do["fp32"] * r for r in ti_le], yy, "|", color="k", ms=24, mew=2.5)
ax.plot([], [], "|", color="k", ms=10, mew=2.5, label="nếu thời gian tỉ lệ với số byte (từ dòng fp32 đầu)")
ax.set_yticks(yy, ten_cach, fontsize=9)
ax.set_xlabel("thời gian một phép nhân ma trận với vector (ms)")
ax.set_xlim(0, 1.25e3 * max(t_do.values()))
ax.set_title("Một tầng 4096 nhân 4096, batch 1, một luồng CPU", fontsize=10)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), fontsize=8, frameon=False)
ax.grid(True, axis="y", alpha=0)
plt.tight_layout()
plt.show()

Đọc hình theo ba câu hỏi.

Thứ nhất, thanh bf16 có chạm gần vạch đen của nó không? Nếu có, phép nhân trên máy này nghẽn ở bộ
nhớ: đọc nửa số byte thì mất gần nửa thời gian, đúng như mô hình lưu lượng của phần 2. Nếu thanh dài
hơn hẳn vạch, máy này không có kernel bf16 nhanh cho phép nhân này và phải đổi từng phần tử sang
fp32 trước khi nhân. Khi đó ít byte hơn không làm nhanh hơn chút nào.

Thứ hai, thanh int8 so với vạch đen của nó, và so với dòng "chỉ riêng đổi kiểu". Trọng số int8 chỉ
bằng một phần tư, nhưng mỗi số nguyên phải được đổi sang số thực trước khi nhân. Nếu riêng việc đổi
kiểu đã tốn ngang cả phép nhân fp32, thì int8 không thể nhanh hơn khi thiếu một kernel nhân thẳng
trên số nguyên. Đó chính là cạm bẫy chương kể: một transformer từ 440 MB xuống 110 MB, mà chi phí
chuyển đổi trên phần cứng không có đường tính độ chính xác thấp nhanh ăn mất phần lợi về độ trễ
(latency).

Thứ ba, hai dòng fp32: cùng 67.1 MB trọng số, cùng phép tính. Nếu chúng chênh nhau, phần chênh chỉ
đến từ kernel. **Tệp nhỏ hơn là điều kiện cần. Tốc độ đến từ kernel khai thác được nó.**

### Cùng tầng ấy, nhiều vector một lúc

Nhân ma trận trọng số với B vector cùng lúc thì mỗi trọng số nạp về được dùng B lần. Mật độ tính
toán (arithmetic intensity), tức số phép tính trên mỗi byte nạp về, tăng theo B: ở fp32 nó vào
khoảng B/2 phép tính mỗi byte. Tới một lúc, phép nhân không còn chờ bộ nhớ mà chờ đơn vị tính toán,
tức là chuyển sang nghẽn ở tính toán (compute-bound). Napkin math 1.3 ở phần 4 nói về chế độ ấy.

### Dự đoán

Thời gian cho **mỗi vector** thay đổi thế nào khi B tăng từ 1 lên 256: giảm đều theo B mãi, hay tới
một lúc thì chững lại?

In [ ]:
CAC_B = [1, 2, 4, 8, 16, 32, 64, 128, 256]
t_lo = []
for B in CAC_B:
    X = torch.randn(B, D)
    t_lo.append(do_thoi_gian(lambda: F.linear(X, W), lap=7))
    mat_do = 2 * D * D * B / (4 * (D * D + 2 * D * B))       # phép tính chia byte đọc và ghi
    do_tren_may(f"B = {B:>3} (mật độ tính toán {mat_do:5.1f} phép/byte): thời gian mỗi vector",
                1e6 * t_lo[-1] / B, "µs")
t_moi_vector = np.array(t_lo) / np.array(CAC_B)
do_tren_may("B = 256 so với B = 1: thời gian mỗi vector giảm", t_moi_vector[0] / t_moi_vector[-1], "lần")

fig, ax = plt.subplots(figsize=(7, 4.3))
ax.loglog(CAC_B, 1e6 * t_moi_vector, "o-", color="C0", lw=1.8, label="đo trên máy này")
ax.loglog(CAC_B, 1e6 * t_moi_vector[0] / np.array(CAC_B), "--", color="0.45",
          label="trần bộ nhớ: cả batch chỉ tốn thời gian đọc trọng số một lần, đo ở B = 1")
ax.axhline(1e6 * t_moi_vector[-1], ls=":", color="C3",
           label="trần tính toán: thời gian mỗi vector đo ở B = 256")
ax.set_xticks(CAC_B, [str(b) for b in CAC_B])
ax.set_xlabel("kích thước batch B (thang log)")
ax.set_ylabel("thời gian mỗi vector (µs, thang log)")
ax.set_title("Tầng 4096 nhân 4096 ở fp32: dùng lại mỗi trọng số B lần", fontsize=10)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

bw_do = cach["fp32"][1] / t_do["fp32"]
do_tren_may("napkin math 1.4 với tốc độ đọc vừa đo thay cho 50 GB/s: token mỗi giây ở FP16",
            bw_do / tong16, "token/s")

Hai đường tham chiếu là hai giới hạn thô, cả hai lấy từ chính phép đo: đường đứt giả định cả batch
chỉ tốn thời gian đọc trọng số một lần, đường chấm là tốc độ tính ở batch lớn nhất. Ở B lớn, đường
đo tiến sát đường chấm và chững lại, vì khi đó đơn vị tính toán mới là giới hạn. Ở B nhỏ, nếu đường
đo bám đường đứt thì thêm vector gần như không tốn thêm thời gian: thời gian là thời gian đọc trọng
số. Nếu nó nằm trên hẳn đường đứt ở vài B nhỏ, thì lại là chuyện kernel như ở hình trước: nhiều
thư viện dùng một kernel nhân ma trận với vector riêng cho B = 1 và một kernel khác cho B từ 2 trở
lên, và hai kernel ấy không nhất thiết nhanh như nhau.

Hai chế độ ấy là ranh giới giữa hai napkin math của chương: 1.4 cho trường hợp nghẽn ở bộ nhớ (sinh
từng token), 1.3 cho trường hợp nghẽn ở tính toán (batch lớn, mật độ tính toán cao).

Dòng cuối là một phép thay số chứ không phải phép đo mô hình 7B: một luồng CPU đọc chậm hơn nhiều
so với cả bộ máy, và máy thật dùng mọi nhân. Nhưng nó cho thấy cùng một phép tính của chương áp được
vào bất kỳ thiết bị nào, miễn là biết băng thông của nó.

## 4. Phía tính toán: thanh ghi, đơn vị INT8 và năng lượng mỗi phép tính

Khi một tầng nghẽn ở tính toán, lợi ích của lượng tử hoá đến từ chỗ khác: phần cứng xếp được nhiều
giá trị nhỏ hơn vào một lệnh, hoặc có đơn vị riêng cho số nguyên. Napkin math 1.2 tính cho thanh ghi
AVX-512 theo kiểu một lệnh, nhiều dữ liệu (SIMD). Napkin math 1.3 lấy tỉ số thông số kỹ thuật của
một bộ tăng tốc (accelerator) lớp A100. Cả hai là phép chia, nên ta chỉ tính lại.

In [ ]:
THANH_GHI = sach(512, nguon="512 bit · 16 · 64")
theo_sach("số phần tử FP32 mỗi lệnh", THANH_GHI // BIT_FP32, sach=16, nguon="512 bit · 16 · 64")
theo_sach("số phần tử INT8 mỗi lệnh", THANH_GHI // BIT_INT8, sach=64, nguon="512 bit · 16 · 64")
theo_sach("INT8 so với FP32", (THANH_GHI // BIT_INT8) / (THANH_GHI // BIT_FP32), sach=4,
          nguon="16 · 64 · 4×")
INT8_TOPS = sach(624, nguon="624 TOPS · 312 TFLOP/s")
FP16_TFLOPS = sach(312, nguon="624 TOPS · 312 TFLOP/s")
theo_sach("INT8 so với FP16 trên A100, tỉ số thông số kỹ thuật", INT8_TOPS / FP16_TFLOPS, sach=2,
          nguon="624 TOPS · 312 TFLOP/s · ~2×", tol=0.05);

Cả hai là trần, không phải kết quả. 4 lần của thanh ghi chỉ đến khi lệnh INT8 chạy nhanh ngang lệnh
FP32 trên phần cứng ấy, còn 2 lần của A100 chỉ đến khi kernel và cách bố trí dữ liệu đi qua đúng
đường INT8. Phần 3 vừa cho thấy chuyện gì xảy ra khi không có đường ấy.

### Thang năng lượng

Hình 14 và bảng 8 của chương cho năng lượng **đại diện** của từng phép, ở công nghệ 45 nm, theo
Horowitz (2014). Các tỉ số của chương đều so với phép cộng INT8.

### Dự đoán

Mô hình 7B ở FP16 của phần 2 sinh một token: phải đọc 14 GB trọng số từ DRAM, và làm chừng một phép
nhân cộng dồn cho mỗi trọng số. Phần nào tốn năng lượng hơn, đọc hay tính, và hơn cỡ bao nhiêu lần?

In [ ]:
pJ = {   # năng lượng đại diện mỗi phép, picojoule
    "cộng INT8": sach(0.03, nguon="0.1 · 0.03 · 0.9 · 0.4 pJ"),
    "cộng INT32": sach(0.1, nguon="0.1 · 0.03 · 0.9 · 0.4 pJ"),
    "nhân INT8": sach(0.2, nguon="≈3.7 pJ · ≈0.2 pJ · 18.5×", tol=0.05),
    "cộng FP16": sach(0.4, nguon="0.1 · 0.03 · 0.9 · 0.4 pJ"),
    "cộng FP32": sach(0.9, nguon="0.1 · 0.03 · 0.9 · 0.4 pJ"),
    "nhân 32 bit": sach(3.7, nguon="≈3.7 pJ · ≈0.2 pJ · 18.5×", tol=0.05),
    "đọc DRAM 32 bit": sach(640, nguon="21,333.3× · 640 pJ"),
}
SRAM = (sach(5, nguon="5–50 pJ · 167–1,667 lần"), sach(50, nguon="5–50 pJ · 167–1,667 lần"))
co_so = pJ["cộng INT8"]
theo_sach("nhân 32 bit so với nhân INT8", pJ["nhân 32 bit"] / pJ["nhân INT8"], sach=18.5,
          nguon="≈0.2 pJ · 18.5×", tol=0.05)
theo_sach("cộng FP32 so với cộng INT8", pJ["cộng FP32"] / co_so, sach=30, nguon="167–1,667 lần · 30×")
theo_sach("đọc SRAM nhỏ so với cộng INT8", SRAM[0] / co_so, sach=167, nguon="5–50 pJ · 167–1,667 lần")
theo_sach("đọc SRAM lớn so với cộng INT8", SRAM[1] / co_so, sach=1667, nguon="5–50 pJ · 167–1,667 lần")
theo_sach("đọc DRAM 32 bit so với cộng INT8", pJ["đọc DRAM 32 bit"] / co_so, sach=21_333.3,
          nguon="30× · 21,333.3×")
mac_fp32 = pJ["nhân 32 bit"] + pJ["cộng FP32"]
mac_int8 = pJ["nhân INT8"] + pJ["cộng INT8"]
theo_sach("một phép nhân cộng dồn FP32 so với INT8", mac_fp32 / mac_int8, sach=20, nguon="khoảng 20×")
print(f"   nếu phép INT8 cộng dồn trong INT32: {mac_fp32 / (pJ['nhân INT8'] + pJ['cộng INT32']):.1f}×")

# Ước lượng của sổ tay cho một token của mô hình 7B ở phần 2: đọc trọng số, mỗi lần 32 bit,
# so với một phép nhân cộng dồn cho mỗi trọng số tính theo giá FP32 (cao hơn giá FP16)
print()
for ten, b in [("FP16", BYTE_FP16), ("INT4", BYTE_INT4)]:
    doc = n * b / 4 * pJ["đọc DRAM 32 bit"] * 1e-12
    tinh = n * mac_fp32 * 1e-12
    print(f"trọng số {ten}: đọc từ DRAM {doc:.2f} J · tính {tinh:.3f} J · đọc gấp {doc / tinh:.0f} lần tính")

# Hình: thang năng lượng, thang log
ten_phep = ["cộng INT8", "cộng INT32", "nhân INT8", "cộng FP16", "cộng FP32", "nhân 32 bit",
            "đọc SRAM", "đọc DRAM 32 bit"]
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for i, t in enumerate(ten_phep):
    trai, phai = (SRAM if t == "đọc SRAM" else (0.01, pJ[t]))
    mau = "C1" if t.startswith("đọc") else "C0"
    ax.barh(i, phai - trai, left=trai, color=mau, height=0.6)
ax.set_yticks(range(len(ten_phep)), ten_phep, fontsize=9)
ax.set_xscale("log")
ax.set_xlim(0.01, 2e4)
for t, nhan in [("cộng INT8", "1× (mốc so sánh)"), ("cộng FP32", "sách: 30×"),
                ("nhân 32 bit", "sách: 18.5× phép nhân INT8"), ("đọc DRAM 32 bit", "sách: 21,333.3×")]:
    ax.text(pJ[t] * 1.4, ten_phep.index(t), nhan, va="center", fontsize=8)
ax.text(SRAM[1] * 1.4, ten_phep.index("đọc SRAM"), "sách: 167–1,667×\n(5–50 pJ, tuỳ cỡ SRAM)",
        va="center", fontsize=8)
ax.plot([], [], "s", color="C0", label="phép tính")
ax.plot([], [], "s", color="C1", label="đọc bộ nhớ")
ax.legend(loc="upper right", fontsize=8)
ax.set_xlabel("năng lượng mỗi phép (pJ, thang log), giá trị đại diện ở 45 nm")
ax.set_title("Thang năng lượng của chương: di chuyển dữ liệu đắt hơn tính", fontsize=10)
ax.grid(True, axis="y", alpha=0)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

"Khoảng 20×" của chương cho một phép nhân cộng dồn khớp đúng với tổng nhân và cộng: 3.7 cộng 0.9 so
với 0.2 cộng 0.03. Dòng ngay dưới cho thấy tỉ số ấy tuỳ độ chính xác của phép cộng dồn: phần cứng
thường cộng dồn tích INT8 trong INT32 để khỏi tràn số, và khi đó tỉ số nhỏ hơn. Chương cũng nhắc
rằng tỉ số của phép tính không phải là dự đoán thời lượng pin.

Phép ước lượng một token nói điều quan trọng hơn: với giá đại diện của chương, đọc trọng số từ DRAM
tốn gấp vài chục lần chính các phép tính trên chúng, kể cả khi tính theo giá FP32. Nên phần lớn năng
lượng lượng tử hoá tiết kiệm được nằm ở số byte không phải di chuyển nữa, không nằm ở phép nhân rẻ
hơn.

## 5. Hợp nhất toán tử: đếm byte của Conv-BN-ReLU

Không hợp nhất, mỗi phép đọc đầu vào của nó từ bộ nhớ ngoài rồi ghi đầu ra về đó. Với một chuỗi $N$
phép trên cùng một tensor trung gian $M$ byte, lưu lượng là $2NM$. Hợp nhất được, và giữ phần trung
gian trên chip, thì còn gần $2M$. Ví dụ của chương: một tầng ResNet-50 có 256 kênh, ánh xạ đặc trưng
(feature map) 28 nhân 28, mỗi giá trị 4 byte, băng thông HBM mô hình hoá là 900 GB/s.

### Dự đoán

Theo công thức, Conv-BN-ReLU giảm số lượt chuyển 3 lần. Khi tính cả trọng số của phép tích chập,
tổng lưu lượng giảm được bao nhiêu phần trăm: gần 67, hay ít hơn hẳn?

In [ ]:
KENH = sach(256, nguon="256 kênh · 28 nhân 28")
CANH = sach(28, nguon="28 nhân 28 · 3.2 MB")
BYTE_GT = BIT_FP32 // 8
N_PHEP = 3
anh_xa = KENH * CANH * CANH * BYTE_GT
theo_sach("một ánh xạ đặc trưng", anh_xa / KB, sach=802.8, nguon="802.8 KB · 2.4 MB")
theo_sach("2NM so với 2M, N = 3", (2 * N_PHEP * anh_xa) / (2 * anh_xa), sach=3, nguon="6 → 2 · 3×")
theo_sach("lưu lượng trung gian được bỏ", 4 * anh_xa / MB, sach=3.2, nguon="28 nhân 28 · 3.2 MB")

w_conv = sach(2.4, nguon="802.8 KB · 2.4 MB") * MB
p_bn = sach(2, nguon="4 MB · 2 KB · 1.6 MB") * KB
luu_luong = {   # đọc vào, tham số, ghi ra; byte
    "Conv": (anh_xa, w_conv, anh_xa),
    "BN": (anh_xa, p_bn, anh_xa),
    "ReLU": (anh_xa, 0, anh_xa),
}
theo_sach("Conv", sum(luu_luong["Conv"]) / MB, sach=4, nguon="2.4 MB · 4 MB · 2 KB")
theo_sach("BN", sum(luu_luong["BN"]) / MB, sach=1.6, nguon="2 KB · 1.6 MB · 7.2 MB")
theo_sach("ReLU", sum(luu_luong["ReLU"]) / MB, sach=1.6, nguon="2 KB · 1.6 MB · 7.2 MB")
chua = sum(sum(v) for v in luu_luong.values())
da = anh_xa + w_conv + anh_xa       # đọc đầu vào và trọng số một lần, ghi đầu ra cuối một lần
theo_sach("tổng, chưa hợp nhất", chua / MB, sach=7.2, nguon="1.6 MB · 7.2 MB")
theo_sach("tổng, đã hợp nhất", da / MB, sach=4, nguon="2.4 MB · 4 MB")
theo_sach("giảm lưu lượng", 100 * (1 - da / chua), sach=44.5, nguon="7.2 MB · 44.5 phần trăm")
r_chua = sach(7.2, nguon="1.6 MB · 7.2 MB")      # hai tổng như sách in
r_da = sach(4, nguon="2.4 MB · 4 MB")
assert chua - da == 4 * anh_xa + p_bn      # đúng bốn lượt trung gian và tham số BN biến mất

HBM = sach(900, nguon="900 GB/s · 8 µs")
t_chua, t_da = chua / (HBM * GB), da / (HBM * GB)
theo_sach("cận dưới của phần lưu lượng, chưa hợp nhất", 1e6 * t_chua, sach=8, nguon="900 GB/s · 8 µs")
theo_sach("cận dưới của phần lưu lượng, đã hợp nhất", 1e6 * t_da, sach=4.5, nguon="8 µs · 4.5 µs · 1.8×")
theo_sach("tỉ số", t_chua / t_da, sach=1.8, nguon="4.5 µs · 1.8×")

SO_BO_BA = sach(53, nguon="53 · 159 · 53")
theo_sach("số lần gọi kernel, chưa hợp nhất", N_PHEP * SO_BO_BA, sach=159, nguon="53 · 159 · 53")

# Hình: lưu lượng tách theo loại dữ liệu
fig, ax = plt.subplots(figsize=(7, 2.6))
nhom = ["đã hợp nhất", "chưa hợp nhất"]
trung_gian = [2 * anh_xa / MB, 2 * N_PHEP * anh_xa / MB]
tham_so = [w_conv / MB, (w_conv + p_bn) / MB]
ax.barh(nhom, trung_gian, color="C0", height=0.55, label="đọc và ghi ánh xạ đặc trưng")
ax.barh(nhom, tham_so, left=trung_gian, color="C1", height=0.55, label="trọng số Conv và tham số BN")
for i, (tong, cua_sach) in enumerate([(da, r_da), (chua, r_chua)]):
    ax.text(tong / MB + 0.1, i, f"{tong / MB:.2f} MB (sách: {cua_sach:g} MB)", va="center", fontsize=8)
ax.text(trung_gian[1] / 2, 1, "6 lượt", ha="center", va="center", color="white", fontsize=8)
ax.text(trung_gian[0] / 2, 0, "2 lượt", ha="center", va="center", color="white", fontsize=8)
ax.set_xlim(0, 9)
ax.set_xlabel("lưu lượng bộ nhớ của một tầng Conv-BN-ReLU (MB)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=8, frameon=False)
ax.grid(True, axis="y", alpha=0)
plt.tight_layout()
plt.show()

Hợp nhất chỉ bỏ được phần màu xanh: phần trung gian giảm đúng 3 lần, còn 2.4 MB trọng số thì phép
tích chập vẫn phải đọc. Vì thế tổng lưu lượng giảm chừng 44.5 phần trăm chứ không phải 67, và
cận dưới thời gian chỉ cải thiện 1.8 lần. Chương nhấn mạnh rằng đó là cận dưới của **phần lưu
lượng**, không phải độ trễ cả tầng: thời gian tính tích chập, bộ nhớ đệm và chi phí gọi kernel
chưa có ở đây. Với 53 bộ ba Conv-BN-ReLU trong một đồ thị giả định, hợp nhất còn bớt 106 lần gọi
kernel.

Các số trên khớp sách khi trọng số là đúng 2.4 MB. Ô dưới cho thấy chúng nhạy thế nào với cách làm
tròn.

In [ ]:
print(f"từ hai số đã làm tròn mà sách in: (7.2 − 4) / 7.2 = {100 * (r_chua - r_da) / r_chua:.1f} phần trăm")

LOI = 3    # giả thuyết của sổ tay: lõi tích chập 3 nhân 3, từ 256 kênh sang 256 kênh
w_3x3 = LOI * LOI * KENH * KENH * BYTE_GT
theo_sach("trọng số lõi 3 nhân 3", w_3x3 / MB, sach=2.4, nguon="802.8 KB · 2.4 MB")
da_3 = 2 * anh_xa + w_3x3
chua_3 = da_3 + sum(luu_luong["BN"]) + sum(luu_luong["ReLU"])
print(f"với trọng số chưa làm tròn {w_3x3 / MB:.3f} MB: giảm {100 * (1 - da_3 / chua_3):.1f} phần trăm, "
      f"cận dưới đã hợp nhất {1e6 * da_3 / (HBM * GB):.2f} µs, tỉ số {chua_3 / da_3:.2f}×")

Ba cách tính cho ba số khác nhau ở chữ số cuối. Chia hai số đã làm tròn cho 44.4; sách in 44.5, ra
từ các giá trị chưa làm tròn với trọng số đúng 2.4 MB. Còn nếu trọng số là của một lõi 3 nhân 3 (giả
thuyết của sổ tay, sách không nói cỡ lõi), nó là 2.36 MB, và phần trăm giảm thành 44.8, cận dưới
thành 4.4 µs. Không cách nào sai; chúng chỉ làm tròn ở những chỗ khác nhau, và kết luận không đổi.

## 6. Định luật Amdahl, và khoảng cách giữa giấy và thực tế

Một yêu cầu đi qua nhiều bước: nạp và giải mã đầu vào, tiền xử lý, mô hình, hậu xử lý, gửi kết quả.
Nén chỉ làm nhanh bước mô hình. Nếu mô hình chiếm phần $p$ của độ trễ cả yêu cầu và nhanh lên $s$
lần, cả yêu cầu nhanh lên

$$S = \frac{1}{(1 - p) + p / s} \;\le\; \frac{1}{1 - p}.$$

Ví dụ của chương: mô hình chiếm 20 phần trăm độ trễ, 80 phần trăm còn lại là nạp dữ liệu, tiền xử lý
và hậu xử lý.

### Dự đoán

Nếu lượng tử hoá INT8 làm mô hình nhanh đúng 4 lần, cả yêu cầu nhanh lên bao nhiêu lần? Còn nếu mô
hình nhanh vô hạn?

In [ ]:
P_MO_HINH = sach(20, nguon="20 phần trăm · 80 phần trăm · 1.25×") / 100


def amdahl(p, s):
    return 1 / ((1 - p) + p / s)


theo_sach("trần tăng tốc khi mô hình nhanh vô hạn", 1 / (1 - P_MO_HINH), sach=1.25,
          nguon="80 phần trăm · 1.25×")
print(f"   mô hình nhanh 4 lần: cả yêu cầu nhanh {amdahl(P_MO_HINH, 4):.2f} lần")
print(f"   mô hình nhanh 1,000 lần: cả yêu cầu nhanh {amdahl(P_MO_HINH, 1000):.3f} lần")
assert amdahl(P_MO_HINH, 1e12) <= 1 / (1 - P_MO_HINH)

s_mh = np.logspace(0, 3, 200)
fig, ax = plt.subplots(figsize=(7, 4.8))
for p, mau, nhan in [(0.9, "C2", "mô hình chiếm 90 % (giả định của sổ tay)"),
                     (0.5, "C0", "mô hình chiếm 50 % (giả định của sổ tay)"),
                     (P_MO_HINH, "C3", "mô hình chiếm 20 % (ví dụ của chương)")]:
    ax.plot(s_mh, amdahl(p, s_mh), color=mau, lw=1.8, label=nhan)
    ax.axhline(1 / (1 - p), color=mau, ls=":", lw=1)
ax.plot([4], [amdahl(P_MO_HINH, 4)], "o", color="C3")
ax.annotate(f"mô hình nhanh 4 lần:\ncả yêu cầu {amdahl(P_MO_HINH, 4):.2f} lần", xy=(4, amdahl(P_MO_HINH, 4)),
            xytext=(9, 2.6), fontsize=8, arrowprops=dict(arrowstyle="->", color="0.3"))
ax.text(1.1e2, 1 / (1 - P_MO_HINH) + 0.15, "sách: trần 1.25×", color="C3", fontsize=8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_yticks([1, 1.25, 2, 5, 10], ["1", "1.25", "2", "5", "10"])
ax.set_xlabel("mô hình nhanh lên s lần (thang log)")
ax.set_ylabel("cả yêu cầu nhanh lên (thang log)")
ax.set_title("Định luật Amdahl: phần ngoài mô hình đặt trần", fontsize=10)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Ở ví dụ của chương, mô hình nhanh 4 lần chỉ làm cả yêu cầu nhanh chưa tới 1.2 lần, và không gì làm
trên mô hình vượt được 1.25. Đó là lý do chương bắt đầu phần đo lường bằng phân tích hiệu năng
(profiling) **toàn bộ** đường đi của một yêu cầu trên phần cứng đích: nếu tiền xử lý chiếm phần lớn,
đó mới là chỗ cần tối ưu.

### 8× trên giấy, 1.5× thực tế, và ba kịch bản mô hình hoá

Kịch bản minh hoạ của chương ghép cắt tỉa 50 phần trăm với lượng tử hoá INT8 thành một mục tiêu tính
nhẩm 8×, rồi **giả định** chỉ đạt được 1.5×. Đó không phải số đo. Nó minh hoạ các nguồn hao hụt mà
các phần trên vừa tính: số 0 lưu dày vẫn tốn băng thông, thiếu kernel cho định dạng mới, chi phí đổi
kiểu, và phần ngoài mô hình. Ba con số khác của chương cũng dễ bị đọc như kết quả đo, và cả ba là
**kịch bản mô hình hoá**, tức là phép kế toán byte chứ không phải một lần triển khai được báo cáo:

* đường ống BERT trên điện thoại (ví dụ 1.3): một mô hình cỡ BERT-Base 110 triệu tham số, từ 440 MB
  xuống 28 MB qua cắt tỉa có cấu trúc, chưng cất tri thức (knowledge distillation) và huấn luyện có
  tính tới lượng tử hoá (QAT), mất 0.6 phần trăm chất lượng;
* cạm bẫy: một transformer từ 440 MB xuống 110 MB mà độ trễ không giảm theo;
* ResNet-50 FP32 so với INT8 (bảng 16).

In [ ]:
cat = sach(50, nguon="50 phần trăm · 8× · 1.5×") / 100
giay = 1 / (1 - cat) * (BIT_FP32 / BIT_INT8)
theo_sach("mục tiêu trên giấy: cắt tỉa 50 % rồi INT8", giay, sach=8, nguon="50 phần trăm · 8× · 1.5×")
thuc = sach(1.5, nguon="8× · 1.5× · 18.8 phần trăm")
theo_sach("phần mục tiêu đạt được", 100 * thuc / giay, sach=18.8, nguon="1.5× · 18.8 phần trăm",
          tol=0.051)   # 18.75, sách làm tròn lên
it_tham_so = sach(40, nguon="70 phần trăm · 40 phần trăm · 12 phần trăm") / 100
giam_tre = sach(12, nguon="40 phần trăm · 12 phần trăm") / 100
print(f"   kịch bản giả định: ít hơn 40 % tham số mà độ trễ chỉ giảm 12 %, tức nhanh {1 / (1 - giam_tre):.2f} lần"
      f" thay vì {1 / (1 - it_tham_so):.2f} lần nếu thời gian tỉ lệ với tham số")

print()
n_bert = sach(110e6, nguon="440 MB · 28 MB · 16× · 110 triệu")
bert_fp32 = n_bert * BIT_FP32 / 8 / MB
theo_sach("cỡ BERT-Base ở FP32", bert_fp32, sach=440, nguon="440 MB · 28 MB · 16×")
bert_nho = sach(28, nguon="440 MB · 28 MB · 16×")
theo_sach("tỉ số dung lượng của đường ống", bert_fp32 / bert_nho, sach=16, nguon="440 MB · 28 MB · 16×")
print(f"   440 MB chia đúng 16 lần là {bert_fp32 / 16:.1f} MB; sách in 28 MB")
theo_sach("cỡ BERT-Base ở INT8 (cạm bẫy)", n_bert * BIT_INT8 / 8 / MB, sach=110, nguon="440 MB · 110 MB")

print()
do_chinh_xac = (sach(76.1, nguon="76.1 · 75.8 · 0.3 điểm"), sach(75.8, nguon="76.1 · 75.8 · 0.3 điểm"))
theo_sach("ResNet-50: mất độ chính xác top-1", do_chinh_xac[0] - do_chinh_xac[1], sach=0.3,
          nguon="76.1 · 75.8 · 0.3 điểm")
kich_thuoc = sach(102.4, nguon="102.4 MB · 25.6 MB · 4×")
theo_sach("ResNet-50 ở INT8", kich_thuoc * BIT_INT8 / BIT_FP32, sach=25.6, nguon="102.4 MB · 25.6 MB · 4×")
do_tre = (sach(1.56, nguon="1.56 ms · 0.39 ms"), sach(0.39, nguon="1.56 ms · 0.39 ms"))
print(f"   độ trễ mô hình hoá trên V100: {do_tre[0]} → {do_tre[1]} ms, tức {do_tre[0] / do_tre[1]:.1f}×")
nang_luong = sach(0.25, nguon="0.25 J · 0.06 J")
theo_sach("năng lượng mỗi lần suy luận ở INT8, nếu giảm đúng 4 lần", nang_luong / 4, sach=0.06,
          nguon="0.25 J · 0.06 J")
hieu_chuan = (sach(2.1, nguon="0.06 J · 2.1 · 3.4 phần trăm"), sach(3.4, nguon="2.1 · 3.4 phần trăm"))
print(f"   sai số hiệu chuẩn (calibration): {hieu_chuan[0]} → {hieu_chuan[1]} phần trăm, tăng dù độ chính xác gần như giữ")
s_resnet = do_tre[0] / do_tre[1]
print(f"   ghép với ví dụ Amdahl: nếu mô hình chỉ chiếm {P_MO_HINH:.0%} độ trễ yêu cầu, "
      f"cả yêu cầu nhanh {amdahl(P_MO_HINH, s_resnet):.2f} lần")

Tỉ số 16× của đường ống BERT là tỉ số làm tròn: 440 chia 16 cho 27.5 MB, sách in 28 MB. Ở bảng 16,
năng lượng INT8 0.06 J là 0.25 chia 4 đã làm tròn, nên chương ghi cả nó là 4×. Đáng để ý hơn là dòng
sai số hiệu chuẩn: độ chính xác top-1 gần như giữ nguyên, trong khi độ tin cậy của mô hình lệch khỏi
tỉ lệ đúng thật nhiều hơn. Một con số độ chính xác không kể hết chuyện.

Hình cuối đặt các tỉ số trên giấy của sổ tay này cạnh kết quả sau khi tính phần không đổi.

In [ ]:
cap = [   # (nhãn, tỉ số trên giấy, tỉ số còn lại, số sách in cho tỉ số còn lại)
    ("Conv-BN-ReLU:\nsố lượt chuyển → lưu lượng", N_PHEP, t_chua / t_da, " (sách: 1.8×)"),
    ("7B, trọng số FP16 → INT4:\nbyte trọng số → token mỗi giây", BYTE_FP16 / BYTE_INT4, t16 / t4,
     " (sách: 2.9×)"),
    ("ResNet-50 INT8 (bảng 16) ghép ví dụ Amdahl:\nđộ trễ mô hình → độ trễ yêu cầu", s_resnet,
     amdahl(P_MO_HINH, s_resnet), " (sổ tay ghép)"),
    ("cắt tỉa 50 % rồi INT8:\ngiấy → giả định của chương", giay, thuc, " (sách: 1.5×)"),
]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
yy = np.arange(len(cap))[::-1]
ax.barh(yy + 0.18, [c[1] for c in cap], height=0.34, color="white", edgecolor="0.3", label="tỉ số trên giấy")
ax.barh(yy - 0.18, [c[2] for c in cap], height=0.34, color="C0", label="sau khi tính phần không đổi")
for y, (_, a, b, cua_sach) in zip(yy, cap):
    ax.text(a + 0.08, y + 0.18, f"{a:g}×", va="center", fontsize=8)
    ax.text(b + 0.08, y - 0.18, f"{b:.2f}×{cua_sach}", va="center", fontsize=8)
ax.set_yticks(yy, [c[0] for c in cap], fontsize=8)
ax.set_xlim(0, 9.5)
ax.set_xlabel("tỉ số (lần)")
ax.set_title("Tỉ số trên giấy và tỉ số còn lại", fontsize=10)
ax.legend(loc="upper right", fontsize=8)
ax.grid(True, axis="y", alpha=0)
plt.tight_layout()
plt.show()

## Thử thêm

1. Ở phần 2, đặt `SO_TOKEN = 32_768` rồi chạy lại ô ấy. Trọng số INT4 còn làm sinh token nhanh
   hơn bao nhiêu lần, và bộ nhớ đệm KV chiếm bao nhiêu phần lưu lượng? (Dấu ✗ khi đó là điều được
   chờ đợi: sách tính ở 4,096 token.)
2. Ở phần 3, đặt `D = 1024`. Ma trận fp32 khi đó chỉ 4 MB và có thể nằm gọn trong bộ nhớ đệm của
   CPU. Thời gian của bf16 và int8 so với fp32 bây giờ thế nào, và vạch "tỉ lệ với byte" còn nói
   đúng không? Thử thêm `KHOI = 4096` để giải lượng tử hoá cả ma trận một lần.
3. Ở phần 6, đặt `P_MO_HINH = 0.6` ngay sau dòng định nghĩa nó. Mô hình phải nhanh bao nhiêu lần
   để cả yêu cầu nhanh gấp đôi? (Dấu ✗ ở dòng trần tăng tốc khi đó là điều được chờ đợi.)

## Tóm lại

* Kích thước mô hình là số tham số nhân số byte mỗi giá trị. Nó quyết định mô hình có vào được thiết
  bị hay không, trước mọi câu hỏi về tốc độ. Bảng 2 chia byte thập phân cho dung lượng nhị phân.
* Khi sinh từng token, thời gian gần bằng số byte phải đọc chia cho băng thông. Lượng tử hoá chỉ
  trọng số tăng tốc ít hơn tỉ lệ nén, vì bộ nhớ đệm KV không nhỏ đi.
* Ít byte hơn chỉ nhanh hơn khi phần cứng tính thẳng trên kiểu nhỏ ấy. Thiếu kernel thì chi phí đổi
  kiểu có thể ăn hết phần lợi.
* Với giá đại diện của chương, đọc DRAM đắt hơn phép tính rất nhiều, nên phần lớn năng lượng tiết
  kiệm được nằm ở byte không phải di chuyển.
* Hợp nhất toán tử chỉ bỏ lưu lượng trung gian, và định luật Amdahl chặn mọi kỹ thuật nén ở phần
  thời gian mà mô hình chiếm. Hãy đo cả đường đi trên phần cứng đích.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Nén mô hình — đổi phần dư thừa lấy ràng buộc triển khai](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch10-model-compression/), dựng từ chương
[*Model Compression*](https://mlsysbook.ai/vol1/model_compression/model_compression.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.